In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Qiskit Aer 프리미티브를 사용한 스태빌라이저 Circuit의 효율적인 시뮬레이션

<details>
<summary><b>패키지 버전</b></summary>

이 페이지의 코드는 다음 요구 사항을 사용하여 개발되었습니다.
이 버전 이상을 사용하시길 권장합니다.

```
qiskit[all]~=2.3.0
qiskit-aer~=0.17
```
</details>
이 페이지에서는 Qiskit Aer 프리미티브를 사용하여 Pauli 노이즈가 적용된 경우를 포함한 스태빌라이저 Circuit을 효율적으로 시뮬레이션하는 방법을 설명합니다.

스태빌라이저 Circuit은 클리퍼드 Circuit(Clifford Circuit)이라고도 하며, 고전적으로 효율적으로 시뮬레이션할 수 있는 중요한 제한적 양자 Circuit 클래스입니다. 스태빌라이저 Circuit을 정의하는 데는 여러 가지 동등한 방법이 있습니다. 한 가지 정의에 따르면, 스태빌라이저 Circuit은 다음 Gate만으로 구성된 양자 Circuit입니다:

- [CX](../api/qiskit/qiskit.circuit.library.CXGate)
- [Hadamard](../api/qiskit/qiskit.circuit.library.HGate)
- [S](../api/qiskit/qiskit.circuit.library.SGate)
- [Measurement](../api/qiskit/circuit#qiskit.circuit.Measure)

Hadamard와 S를 사용하면 각도가 집합 ${0, \frac{\pi}{2}, \pi, \frac{3\pi}{2}}$에 포함되는 Pauli 회전 Gate([$R_x$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RXGate), [$R_y$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RYGate), [$R_z$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RZGate))를 구성할 수 있으므로(전역 위상 제외), 이러한 Gate도 정의에 포함할 수 있습니다.

스태빌라이저 Circuit은 양자 오류 수정 연구에 중요합니다. 고전적으로 시뮬레이션 가능하다는 특성 덕분에 양자 컴퓨터의 출력을 검증하는 데도 유용합니다. 예를 들어, 양자 컴퓨터에서 100개의 Qubit을 사용하는 양자 Circuit을 실행하려 한다고 가정해 봅시다. 양자 컴퓨터가 올바르게 동작하고 있는지 어떻게 알 수 있을까요? 100개의 Qubit을 사용하는 양자 Circuit은 무차별 대입 방식의 고전적 시뮬레이션으로는 처리할 수 없는 범위입니다. Circuit을 스태빌라이저 Circuit으로 변경하면, 원하는 Circuit과 유사한 구조를 가지면서도 고전적 컴퓨터에서 시뮬레이션할 수 있는 Circuit을 양자 컴퓨터에서 실행할 수 있습니다. 스태빌라이저 Circuit에서 양자 컴퓨터의 출력을 확인함으로써, 비스태빌라이저 Circuit에서도 올바르게 동작하고 있다는 신뢰를 얻을 수 있습니다. 이 아이디어의 실제 적용 사례는 [*Evidence for the utility of quantum computing before fault tolerance*](https://www.nature.com/articles/s41586-023-06096-3)를 참고하세요.

[Qiskit Aer 프리미티브를 사용한 정확한 시뮬레이션 및 노이즈 시뮬레이션](simulate-with-qiskit-aer)에서는 [Qiskit Aer](https://qiskit.org/ecosystem/aer/)를 사용하여 일반적인 양자 Circuit의 정확한 시뮬레이션과 노이즈 시뮬레이션을 수행하는 방법을 설명합니다. 해당 문서에서 사용된 예제 Circuit인 [efficient_su2](../api/qiskit/qiskit.circuit.library.efficient_su2)로 구성된 8-Qubit Circuit을 살펴보겠습니다:

In [1]:
from qiskit.circuit.library import efficient_su2

n_qubits = 8
circuit = efficient_su2(n_qubits)
circuit.draw("mpl")

<Image src="../docs/images/guides/simulate-stabilizer-circuits/extracted-outputs/2d26ac3e-2a6a-4d73-900f-470200a63154-0.svg" alt="Output of the previous code cell" />

![이전 코드 셀의 출력](../docs/images/guides/simulate-stabilizer-circuits/extracted-outputs/2d26ac3e-2a6a-4d73-900f-470200a63154-0.svg)

Qiskit Aer를 사용하면 이 Circuit을 쉽게 시뮬레이션할 수 있었습니다. 그런데 Qubit 수를 500으로 설정한다고 가정해 봅시다:

In [2]:
n_qubits = 500
circuit = efficient_su2(n_qubits)
# don't try to draw the circuit because it's too large

양자 Circuit 시뮬레이션 비용은 Qubit 수에 따라 지수적으로 증가하므로, 이처럼 큰 Circuit은 Qiskit Aer와 같은 고성능 시뮬레이터의 능력도 초과하게 됩니다. 일반적으로 Qubit 수가 약 50~100개를 초과하면 일반적인 양자 Circuit의 고전적 시뮬레이션이 불가능해집니다. 단, efficient_su2 Circuit은 $R_y$ 및 $R_z$ Gate의 각도로 매개변수화되어 있습니다. 이 모든 각도가 집합 ${0, \frac{\pi}{2}, \pi, \frac{3\pi}{2}}$에 포함된다면, 해당 Circuit은 스태빌라이저 Circuit이 되어 효율적으로 시뮬레이션할 수 있습니다!

다음 셀에서는 Circuit이 스태빌라이저 Circuit임을 보장하도록 무작위로 선택된 매개변수를 사용하여, 스태빌라이저 Circuit 시뮬레이터를 Backend로 하는 Sampler 프리미티브로 Circuit을 실행합니다.

In [3]:
import numpy as np
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import SamplerV2 as Sampler

measured_circuit = circuit.copy()
measured_circuit.measure_all()

rng = np.random.default_rng(1234)
params = rng.choice(
    [0, np.pi / 2, np.pi, 3 * np.pi / 2],
    size=circuit.num_parameters,
)

# Initialize a Sampler backed by the stabilizer circuit simulator
exact_sampler = Sampler(
    options=dict(backend_options=dict(method="stabilizer"))
)
# The circuit needs to be transpiled to the AerSimulator target
pass_manager = generate_preset_pass_manager(
    1, AerSimulator(method="stabilizer")
)
isa_circuit = pass_manager.run(measured_circuit)
pub = (isa_circuit, params)
job = exact_sampler.run([pub])
result = job.result()
pub_result = result[0]
counts = pub_result.data.meas.get_counts()

스태빌라이저 Circuit 시뮬레이터는 노이즈 시뮬레이션도 지원하지만, 제한된 종류의 노이즈 모델에 한해서만 가능합니다. 구체적으로, 모든 양자 노이즈는 [Pauli 오류](https://qiskit.org/ecosystem/aer/stubs/qiskit_aer.noise.pauli_error.html#qiskit_aer.noise.pauli_error) 채널로 특성화되어야 합니다. [디폴라라이징 오류](https://qiskit.org/ecosystem/aer/stubs/qiskit_aer.noise.depolarizing_error.html)는 이 범주에 해당하므로 시뮬레이션할 수 있습니다. [판독 오류](https://qiskit.org/ecosystem/aer/stubs/qiskit_aer.noise.ReadoutError.html)와 같은 고전적 노이즈 채널도 시뮬레이션할 수 있습니다.

다음 코드 셀은 이전과 동일한 시뮬레이션을 실행하지만, 이번에는 각 CX Gate에 2%의 디폴라라이징 오류를 추가하고 측정된 각 비트가 5% 확률로 뒤집히는 판독 오류를 추가하는 노이즈 모델을 지정합니다.

In [4]:
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError

noise_model = NoiseModel()
cx_depolarizing_prob = 0.02
bit_flip_prob = 0.05
noise_model.add_all_qubit_quantum_error(
    depolarizing_error(cx_depolarizing_prob, 2), ["cx"]
)
noise_model.add_all_qubit_readout_error(
    ReadoutError(
        [
            [1 - bit_flip_prob, bit_flip_prob],
            [bit_flip_prob, 1 - bit_flip_prob],
        ]
    )
)

noisy_sampler = Sampler(
    options=dict(
        backend_options=dict(method="stabilizer", noise_model=noise_model)
    )
)
job = noisy_sampler.run([pub])
result = job.result()
pub_result = result[0]
counts = pub_result.data.meas.get_counts()

이제 스태빌라이저 시뮬레이터를 Backend로 하는 Estimator 프리미티브를 사용하여 관측량 $ZZ \cdots Z$의 기댓값을 계산해 보겠습니다. 스태빌라이저 Circuit의 특수한 구조 덕분에, 결과는 0이 될 가능성이 매우 높습니다.

In [5]:
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.primitives import EstimatorV2 as Estimator

observable = SparsePauliOp("Z" * n_qubits)

exact_estimator = Estimator(
    options=dict(backend_options=dict(method="stabilizer")),
)
isa_circuit = pass_manager.run(circuit)
pub = (isa_circuit, observable, params)
job = exact_estimator.run([pub])
result = job.result()
pub_result = result[0]
exact_value = float(pub_result.data.evs)
exact_value

0.0

## Next steps

<Admonition type="tip" title="Recommendations">
  - To simulate circuits with Qiskit Aer, see [Exact and noisy simulation with Qiskit Aer primitives](./simulate-with-qiskit-sdk-primitives).
  - Review the [Qiskit Aer](https://qiskit.org/ecosystem/aer/) documentation.
</Admonition>